# Deep Research Agent

Interactive notebook for running the deep research agent with real LLMs and search.

## Setup

In [1]:
import os

# Set your API key here or via environment variable
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["TAVILY_API_KEY"] = "tvly-..."

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first"

In [2]:
from langchain_anthropic import ChatAnthropic

from deep_research import (
    ClarificationNeeded, Complete, Config,
    DeepResearchAgent, DraftReady, GapReview, PlanReady,
)
from deep_research.events import ResearchUpdate
from deep_research.providers.duckduckgo import DuckDuckGoSearchProvider

fast_llm     = ChatAnthropic(model="claude-haiku-4-5-20251001")
powerful_llm = ChatAnthropic(model="claude-sonnet-4-6")

print("LLMs ready")

LLMs ready


## Configure the agent

In [3]:
config = Config(
    max_research_loops=2,
    breadth=3,                  # search queries per loop
    enable_clarification=False, # set True to be prompted for clarifying questions
    enable_gap_review=False,    # set True to approve/redirect gap analysis
    enable_draft_review=False,  # set True to review draft before finalising
)

agent = DeepResearchAgent(
    fast_llm=fast_llm,
    powerful_llm=powerful_llm,
    search_provider=DuckDuckGoSearchProvider(),
    config=config,
)

print(f"Agent ready  (thread: {agent.thread_id})")

Agent ready  (thread: 3a25e964-8153-4247-b3e6-856fd552ebf0)


## Run a query

Change `QUERY` and re-run this cell.

In [ ]:
QUERY = "How do I set up a python venv for my new project"

result = None
print(f"Query: {QUERY}\n{'='*60}")

async for event in agent.astream(QUERY):
    if isinstance(event, PlanReady):
        print(f"\n[Plan] {len(event.sub_questions)} sub-questions:")
        for sq in event.sub_questions:
            print(f"  • {sq.question}")

    elif isinstance(event, ResearchUpdate):
        print(f"\n[Loop {event.loop_count}] {event.sources_count} sources, {event.findings_count} findings")

    elif isinstance(event, ClarificationNeeded):
        print(f"\n[Clarification needed]")
        answers = {}
        for q in event.questions:
            answers[q] = input(f"  {q}: ")
        await agent.resume(answers)

    elif isinstance(event, GapReview):
        print(f"\n[Gap review] Missing: {event.gaps}")
        resp = input("  Approve gaps / provide custom queries: ")
        await agent.resume(resp or "approve")

    elif isinstance(event, DraftReady):
        print(f"\n[Draft ready] ({len(event.draft)} chars)")
        resp = input("  Approve or give feedback: ")
        await agent.resume(resp or "approve")

    elif isinstance(event, Complete):
        result = event.result
        m = result.metadata
        print(f"\n{'='*60}")
        print(f"Done — {m['loops_run']} loops, {m['total_sources']} sources, {m['elapsed_seconds']}s")

# drain pending async callbacks
import asyncio; await asyncio.sleep(0)

Query: How do I set up a python venv for my new project


## Report

In [ ]:
from IPython.display import Markdown, display

if result:
    display(Markdown(result.final_report))
else:
    print("No result yet — run the query cell first.")

## Sources

In [ ]:
if result:
    for url, src in result.sources.items():
        print(f"[{src.relevance_score:.2f}] {src.title}")
        print(f"        {url}")

## Inspect state

In [ ]:
state = await agent.aget_state()
print(f"loops run  : {state['research_loop_count']}")
print(f"findings   : {len(state['findings'])}")
print(f"sources    : {len(state['sources'])}")
print(f"is_sufficient: {state['is_sufficient']}")